# Stage 1 — Baseline Circuit Latent Space

**Goal:** Train a standard ResNet18 on CIFAR-10 *without* any consistency loss.
Attach the meta-encoder with a standard NT-Xent contrastive objective and visualise
the circuit latent space via UMAP.

**Question being answered:** How much semantic structure exists in the circuit space
of a *normally-trained* model — before any interpretability pressure is applied?

**Outputs from this stage:**
- Trained baseline checkpoint at `experiments/baseline/best.pt`
- UMAP plot of circuit space vs output space
- Silhouette score (cluster separability baseline)
- These numbers are the comparison target for Stage 2

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/YOUR_USERNAME/model_interpretability.git'  # <-- edit this once
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({"Colab" if IN_COLAB else "local"})")  


In [ ]:
import torch
import yaml
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

In [ ]:
with open('configs/baseline.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False))

## 1. Train the Baseline Model

This trains ResNet18 on CIFAR-10 with cross-entropy only (λ=0, no consistency loss).
Expected training time: ~20 min on a single GPU.

**Skip this cell if you already have a checkpoint** at `experiments/baseline/best.pt`.

In [ ]:
from training.trainer import Trainer

trainer = Trainer(config)
trainer.train()

## 2. Load Trained Model

In [ ]:
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder

CHECKPOINT = 'experiments/baseline/best.pt'

mcfg = config['model']
ecfg = mcfg['meta_encoder']
tcfg = config['training']

soft_mask = SoftMask(init_temperature=tcfg['temperature']['init']).to(DEVICE)
backbone = CTLSBackbone(
    arch=mcfg['arch'],
    num_classes=mcfg['num_classes'],
    soft_mask=soft_mask,
    pretrained=False,
).to(DEVICE)
meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    hidden_dim=ecfg.get('hidden_dim', 256),
    embedding_dim=ecfg.get('embedding_dim', 64),
    encoder_type=ecfg.get('encoder_type', 'mlp'),
).to(DEVICE)

ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
backbone.load_state_dict(ckpt['backbone_state'])
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
backbone.eval()
meta_encoder.eval()

print(f"Loaded checkpoint (epoch {ckpt['epoch']}, val_acc={ckpt['val_acc']:.3f})")
print(f"Trajectory dims: {backbone.layer_dims}")
print(f"Number of trajectory points (layers): {len(backbone.layer_dims)}")

## 3. Validation Accuracy

In [ ]:
import torch.nn.functional as F
from data.cifar import get_standard_loaders

dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'],
    batch_size=dcfg['batch_size'],
    num_workers=dcfg.get('num_workers', 4),
    augment=False,
    download=True,
)

correct, total = 0, 0
with torch.no_grad():
    for x, labels in val_loader:
        x, labels = x.to(DEVICE), labels.to(DEVICE)
        logits, _ = backbone(x)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)

print(f'Baseline validation accuracy: {correct/total:.4f} ({100*correct/total:.2f}%)')

## 4. UMAP of Circuit Latent Space vs Output Space

This is the core visualisation for Stage 1. We expect the circuit space to show
*some* semantic clustering (the model did learn to classify), but likely less
tight and less well-separated than after CTLS training in Stage 2.

In [ ]:
from evaluation.circuit_viz import CircuitVisualizer

viz = CircuitVisualizer(backbone, meta_encoder, val_loader, DEVICE)

fig = viz.plot_umap(
    title='Stage 1 — Baseline (no consistency loss)',
    compare_output=True,
    max_samples=5000,
)
os.makedirs('experiments/baseline', exist_ok=True)
fig.savefig('experiments/baseline/umap_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/baseline/umap_baseline.png')

## 5. Cluster Separability (Silhouette Score)

Silhouette score ranges from -1 to 1. Higher = tighter, better-separated clusters.
This becomes our Stage 1 baseline number. Stage 2 should exceed it.

In [ ]:
scores = viz.cluster_separation_score(max_samples=5000)

print('=== Silhouette Scores (Stage 1 Baseline) ===')
print(f"  Circuit space:  {scores['silhouette_circuit']:.4f}")
print(f"  Output space:   {scores['silhouette_output']:.4f}")
print(f"  Delta (C - O):  {scores['delta']:+.4f}")
print()
print('Save these numbers — Stage 2 results will be compared against them.')

## 6. t-SNE (Optional — Slower, Different Geometry)

In [ ]:
fig_tsne = viz.plot_tsne(
    title='Stage 1 — Baseline (t-SNE)',
    compare_output=True,
    max_samples=2000,
)
fig_tsne.savefig('experiments/baseline/tsne_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Per-Layer Trajectory Analysis

Look at where semantic structure *emerges* across layers. This helps set intuition
for what the depth-weighted consistency loss should be targeting.

In [ ]:
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Collect per-layer activations
all_layers = [[] for _ in range(len(backbone.layer_dims))]
all_labels = []
n = 0

with torch.no_grad():
    for x, labels in val_loader:
        x = x.to(DEVICE)
        _, traj = backbone(x)
        for i, h in enumerate(traj):
            all_layers[i].append(h.cpu())
        all_labels.append(labels)
        n += x.shape[0]
        if n >= 2000:
            break

all_layers = [torch.cat(l, dim=0)[:2000].numpy() for l in all_layers]
all_labels = torch.cat(all_labels, dim=0)[:2000].numpy()

# Silhouette per layer (PCA to 50D first for speed)
layer_silhouettes = []
for i, acts in enumerate(all_layers):
    if acts.shape[1] > 50:
        acts_pca = PCA(n_components=50).fit_transform(acts)
    else:
        acts_pca = acts
    sil = silhouette_score(acts_pca, all_labels, metric='euclidean', sample_size=1000)
    layer_silhouettes.append(sil)
    print(f'  Layer {i+1:2d} (dim={backbone.layer_dims[i]:4d}): silhouette={sil:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(layer_silhouettes)+1), layer_silhouettes, 'o-', color='steelblue', lw=2)
ax.set_xlabel('Layer index')
ax.set_ylabel('Silhouette score')
ax.set_title('Baseline: semantic structure emergence across layers')
ax.axhline(0, color='gray', linestyle='--', lw=0.8)
ax.set_xticks(range(1, len(layer_silhouettes)+1))
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/baseline/per_layer_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiments/baseline/per_layer_silhouette.png')

## Summary

Record the key numbers from this stage before proceeding to Stage 2:

| Metric | Stage 1 value |
|--------|---------------|
| Val accuracy | ___ |
| Circuit space silhouette | ___ |
| Output space silhouette | ___ |
| Peak per-layer silhouette (layer ___) | ___ |

**Next:** Run `stage2_ctls.ipynb` to train with the full CTLS objective and compare.